# Part 1: Data Extraction, Cleaning & Database
**Source:** https://en.wikipedia.org/wiki/List_of_highest-grossing_films

**Pipeline:**
1. Fetch main table → rank, title, box_office, year
2. Fetch each film's infobox → director, country
3. Clean all columns
4. Save to SQLite
5. Export to JSON for GitHub Pages

## 0. Imports

In [ ]:
# !pip install requests beautifulsoup4 lxml pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import os
import json

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
BASE    = 'https://en.wikipedia.org'
print('OK')

OK


## 1. Fetch Main Table

In [76]:
URL  = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'
resp = requests.get(URL, headers=HEADERS, timeout=15)
assert resp.status_code == 200

soup  = BeautifulSoup(resp.text, 'lxml')
table = soup.find('table', class_='wikitable')

cols = [th.get_text(strip=True) for th in table.find('tr').find_all(['th','td'])]

In [77]:
# Adjust if column order differs from what's printed above
COL_RANK, COL_TITLE, COL_BOX_OFFICE, COL_YEAR = 0, 2, 3, 4

def cell_text(cells, idx):
    if idx >= len(cells): return None
    t = cells[idx].get_text(separator=' ', strip=True)
    return re.sub(r'\[[^\]]*\]', '', t).strip() or None

rows_data = []
for row in table.find_all('tr')[1:]:
    cells = row.find_all(['td','th'])
    if len(cells) < 4: continue
    wiki_url = wiki_slug = None
    if COL_TITLE < len(cells):
        for a in cells[COL_TITLE].find_all('a', href=True):
            if a['href'].startswith('/wiki/'):
                wiki_url  = BASE + a['href']
                wiki_slug = a['href'].replace('/wiki/', '')
                break

    rows_data.append({
        'rank':       cell_text(cells, COL_RANK),
        'title':      cell_text(cells, COL_TITLE),
        'box_office': cell_text(cells, COL_BOX_OFFICE),
        'year':       cell_text(cells, COL_YEAR),
        'wiki_url':   wiki_url,
        'wiki_slug':  wiki_slug,
    })

df = pd.DataFrame(rows_data)
print(f'Rows: {len(df)}')
df.head()

Rows: 50


,rank,title,box_office,year,wiki_url,wiki_slug
0,1,Avatar,"$2,923,710,708",2009,https://en.wikipedia.org/wiki/Avatar_(2009_film),Avatar_(2009_film)
1,2,Avengers: Endgame,"$2,797,501,328",2019,https://en.wikipedia.org/wiki/Avengers:_Endgame,Avengers:_Endgame
2,3,Avatar: The Way of Water,"$2,334,484,620",2022,https://en.wikipedia.org/wiki/Avatar:_The_Way_...,Avatar:_The_Way_of_Water
3,4,Titanic,"T $2,257,906,828",1997,https://en.wikipedia.org/wiki/Titanic_(1997_film),Titanic_(1997_film)
4,5,Ne Zha 2,"NZ $2,215,690,000",2025,https://en.wikipedia.org/wiki/Ne_Zha_2,Ne_Zha_2


## 2. Scrape Director & Country from Each Film's Infobox

Wikipedia infoboxes have varying structure. The `get_infobox_field` function
tries three strategies in order of reliability:

| Priority | Strategy | Why |
|---|---|---|
| 1 | `<a>` link texts (skip `#` anchors) | Most films — director has a Wikipedia article |
| 2 | `<li>` item texts | Some films list directors in `<ul><li>` without links |
| 3 | Raw `.get_text()` | Fallback for plain text cells |

In [ ]:
import re
import requests
import time
from bs4 import BeautifulSoup

# Обязательно представляемся браузером, чтобы избежать 403 ошибок
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_infobox_field(td):
    """Очищает текст ячейки от мусора и склеивания."""
    for tag in td.find_all(['sup', 'style']):
        tag.decompose()
    for span in td.find_all('span', style=lambda v: v and 'display:none' in v.replace(' ', '')):
        span.decompose()

    for br in td.find_all(['br', 'hr']):
        br.replace_with('\n')
    for li in td.find_all('li'):
        li.insert_after('\n')

    raw_text = td.get_text(separator='\n', strip=True)
    
    seen = set()
    out = []
    for line in raw_text.split('\n'):
        line = re.sub(r'\[[^\]]*\]', '', line)  
        line = line.replace('\xa0', ' ').replace('\u200b', '').strip()
        line = re.sub(r'\s+', ' ', line)
        
        if line and line not in seen:
            seen.add(line)
            out.append(line)

    # Если ничего не нашли, возвращаем строку "Unknown" вместо None
    return ', '.join(out) if out else "Unknown"


def scrape_infobox(url):
    """Ищет режиссера и страну, отсекая любые None."""
    # Если URL пустой, сразу отдаем заглушки
    if not url or str(url).lower() == 'nan':
        return "Unknown", "Unknown"
        
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            print(f"[{r.status_code}] Ошибка доступа к {url}")
            return "Unknown", "Unknown"
            
        page = BeautifulSoup(r.text, 'lxml')
        
        # Ищем ЛЮБЫЕ таблицы, в классе которых есть слово infobox (игнорируя регистр)
        infoboxes = page.find_all('table', class_=re.compile(r'infobox', re.I))
        
        director = "Unknown"
        country = "Unknown"
        
        for infobox in infoboxes:
            for tr in infobox.find_all('tr'):
                # Ищем любые заголовки
                th = tr.find(['th', 'td'], class_=re.compile(r'infobox-label', re.I)) or tr.find('th')
                td = tr.find('td', class_=re.compile(r'infobox-data', re.I)) or tr.find('td')
                
                if not th or not td:
                    continue
                
                label = th.get_text(separator=' ', strip=True).lower().replace('\xa0', ' ')
                
                # Ловим direct, directed, director, directors, filmmaker
                if ('direct' in label or 'filmmaker' in label) and director == "Unknown":
                    director = get_infobox_field(td)
                    
                # Ловим country, countries, origin, nationality
                if ('countr' in label or 'origin' in label or 'nation' in label) and country == "Unknown":
                    country = get_infobox_field(td)
            
            # Если оба поля заполнены (не Unknown), прекращаем поиск
            if director != "Unknown" and country != "Unknown":
                break
                
        return director, country
    except Exception as e:
        print(f'{url}: {e}')
        return "Unknown", "Unknown"


In [88]:
# Создаем пустые списки для новых колонок
directors = []
countries = []

print("Начинаем сбор...")
for i, row in df.iterrows():
    d, c = scrape_infobox(row['wiki_url'])
    directors.append(d)
    countries.append(c)
    
    # Печатаем прогресс
    print(f"[{i+1}/{len(df)}] {row['title']} ... dir={d!r}  country={c!r}")
    
    # Спим 1 секунду между запросами, чтобы не бесить Википедию
    time.sleep(1)

# Записываем результаты обратно в датафрейм
df['director'] = directors
df['country'] = countries


Начинаем сбор...
[1/50] Avatar ... dir='James Cameron'  country='United States, United Kingdom'
[2/50] Avengers: Endgame ... dir='Anthony Russo, Joe Russo'  country='United States'
[3/50] Avatar: The Way of Water ... dir='James Cameron'  country='United States'
[4/50] Titanic ... dir='James Cameron'  country='United States'
[5/50] Ne Zha 2 ... dir='Jiaozi'  country='China'
[6/50] Star Wars: The Force Awakens ... dir='J. J. Abrams'  country='United States'
[7/50] Avengers: Infinity War ... dir='Anthony Russo, Joe Russo'  country='United States'
[8/50] Spider-Man: No Way Home ... dir='Jon Watts'  country='United States'
[9/50] Zootopia 2 † ... dir='Jared Bush, Byron Howard'  country='United States'
[10/50] Inside Out 2 ... dir='Kelsey Mann'  country='United States'
[11/50] Jurassic World ... dir='Colin Trevorrow'  country='United States'
[12/50] The Lion King ... dir='Jon Favreau'  country='United States'
[13/50] The Avengers ... dir='Joss Whedon'  country='United States'
[14/50] Furious

## 3. Clean All Columns

In [89]:
df_clean = df.drop(columns=['wiki_url', 'wiki_slug']).copy()

# rank & year → integer
df_clean['rank'] = pd.to_numeric(df_clean['rank'], errors='coerce').astype('Int64')
df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce').astype('Int64')

# title: remove dagger † (Wikipedia marks for estimated figures)
df_clean['title'] = df_clean['title'].str.replace(r'\s*†\s*', '', regex=True).str.strip()

# box_office: strip prefix abbreviations like 'T $', 'NZ $', 'F8 $'
# Strategy: find the first '$' and parse digits+commas after it
def parse_box_office(val):
    if pd.isna(val): return None
    m = re.search(r'\$([\d,]+)', str(val))
    return float(m.group(1).replace(',', '')) if m else None

df_clean['box_office'] = df_clean['box_office'].apply(parse_box_office)

# country & director: remove empty comma fragments, deduplicate
def clean_list_field(val):
    if pd.isna(val) or val is None: return None
    parts = [p.strip() for p in str(val).split(',')]
    seen, unique = set(), []
    for p in parts:
        if p and p not in seen:
            seen.add(p)
            unique.append(p)
    return ', '.join(unique) or None

df_clean['country']  = df_clean['country'].apply(clean_list_field)
df_clean['director'] = df_clean['director'].apply(clean_list_field)

df_clean = df_clean[['rank', 'title', 'year', 'director', 'box_office', 'country']]


In [90]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 60)
display(df_clean)

,rank,title,year,director,box_office,country
0,1,Avatar,2009,James Cameron,2.923711e+09,"United States, United Kingdom"
1,2,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2.797501e+09,United States
2,3,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States
3,4,Titanic,1997,James Cameron,2.257907e+09,United States
4,5,Ne Zha 2,2025,Jiaozi,2.215690e+09,China
5,6,Star Wars: The Force Awakens,2015,J. J. Abrams,2.068224e+09,United States
6,7,Avengers: Infinity War,2018,"Anthony Russo, Joe Russo",2.048360e+09,United States
7,8,Spider-Man: No Way Home,2021,Jon Watts,1.922599e+09,United States
8,9,Zootopia 2,2025,"Jared Bush, Byron Howard",1.866590e+09,United States
9,10,Inside Out 2,2024,Kelsey Mann,1.698864e+09,United States


inserting missing values

In [82]:
df.loc[24, 'director'] = 'J. A. Bayona'
df.loc[24, 'country'] = 'United States'

In [83]:
df.loc[38, 'director'] = 'Michael Bay'
df.loc[38,'country'] = 'United States'

In [84]:
display(df_clean)

,rank,title,year,director,box_office,country
0,1,Avatar,2009,James Cameron,2.923711e+09,"United States, United Kingdom"
1,2,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2.797501e+09,United States
2,3,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States
3,4,Titanic,1997,James Cameron,2.257907e+09,United States
4,5,Ne Zha 2,2025,Jiaozi,2.215690e+09,China
5,6,Star Wars: The Force Awakens,2015,J. J. Abrams,2.068224e+09,United States
6,7,Avengers: Infinity War,2018,"Anthony Russo, Joe Russo",2.048360e+09,United States
7,8,Spider-Man: No Way Home,2021,Jon Watts,1.922599e+09,United States
8,9,Zootopia 2,2025,"Jared Bush, Byron Howard",1.866590e+09,United States
9,10,Inside Out 2,2024,Kelsey Mann,1.698864e+09,United States


In [85]:
%pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [91]:
from sqlalchemy import create_engine

# 1. Connetion
db_user = 'postgres'
db_password = ''
db_host = 'localhost'
db_port = '5432'
db_name = 'films'

connection_string = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)

# 2.# 2. Download dataframe to PostgreAdmin
try:
    df_clean.to_sql('films', engine, if_exists='append', index=False)
except Exception as e:
    print(f"Erro: {e}")

Erro: (psycopg2.errors.UniqueViolation) ОШИБКА:  повторяющееся значение ключа нарушает ограничение уникальности "films_pkey"
DETAIL:  Ключ "(rank)=(1)" уже существует.

[SQL: INSERT INTO films (rank, title, year, director, box_office, country) VALUES (%(rank__0)s, %(title__0)s, %(year__0)s, %(director__0)s, %(box_office__0)s, %(country__0)s), (%(rank__1)s, %(title__1)s, %(year__1)s, %(director__1)s, %(box_office__1)s, %(c ... 4664 characters truncated ... , (%(rank__49)s, %(title__49)s, %(year__49)s, %(director__49)s, %(box_office__49)s, %(country__49)s)]
[parameters: {'director__0': 'James Cameron', 'rank__0': 1, 'title__0': 'Avatar', 'box_office__0': 2923710708.0, 'country__0': 'United States, United Kingdom', 'year__0': 2009, 'director__1': 'Anthony Russo, Joe Russo', 'rank__1': 2, 'title__1': 'Avengers: Endgame', 'box_office__1': 2797501328.0, 'country__1': 'United States', 'year__1': 2019, 'director__2': 'James Cameron', 'rank__2': 3, 'title__2': 'Avatar: The Way of Water', 'box_o

## 5. Export to JSON (for GitHub Pages)

In [92]:
df_clean.to_json('films.json', orient='records', force_ascii=False, indent=2)
print('Exported → films.json')

with open('films.json') as f:
    data = json.load(f)
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Exported → films.json
{
  "rank": 1,
  "title": "Avatar",
  "year": 2009,
  "director": "James Cameron",
  "box_office": 2923710708.0,
  "country": "United States, United Kingdom"
}
